Qtd de dutos
tlen de dutos
tlen do ac (separado por tipo de vala)
tlen do dc

In [20]:
from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - BOQ.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - BOQ.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - BOQ.dwg")

conn = load_to_duckdb(dxf_file, "PV Paiva")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

1
Project PV Paiva already exists
Removed existing data for project: PV Paiva
Successfully loaded project
Loaded 40737 entities for project: PV Paiva
Don't forget to close connection.
Data Loaded
DuckDB Loaded


In [21]:
cable_data = conn.sql("SELECT layer, name, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name='PV Paiva'")
df = cable_data.df()
conn.close()

print(df.describe())
print(df.info())
print(df['layer'].value_counts())

            start_x        start_y         end_x          end_y        length
count  40737.000000   40737.000000  40737.000000   40737.000000  40737.000000
mean   35061.878906  128095.242188  35061.917969  128104.781250     54.851696
std      597.953857     958.361267    597.325745     957.024414     53.722980
min    33667.550781  125994.796875  33669.246094  125994.796875      0.000000
25%    34616.394531  127314.492188  34620.054688  127272.125000      6.300000
50%    35003.191406  128067.625000  34996.957031  128067.625000     44.631733
75%    35492.964844  128981.703125  35502.054688  128995.648438     92.256584
max    36625.890625  129808.226562  36624.191406  129708.945312    454.309540
<class 'pandas.DataFrame'>
RangeIndex: 40737 entries, 0 to 40736
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   layer    40737 non-null  str    
 1   name     0 non-null      object 
 2   start_x  40737 non-null  float32
 3   start_

In [22]:
mask_ac_cables = (df['layer'] == "EMF_bt_vala_btca_01xC") | (df['layer'] == "EMF_bt_vala_btca_02xC") | (df['layer'] == "EMF_bt_vala_btca_03xC") | (df['layer'] == "EMF_bt_vala_btca_04xC") | (df['layer'] == "EMF_bt_vala_btca_05xC") | (df['layer'] == "EMF_bt_vala_btca_06xC") | (df['layer'] == "EMF_bt_vala_btca_07xC") | (df['layer'] == "EMF_bt_vala_btca_08xC")

mask_ac_watercourse = (df['layer'] == "EMF_bt_watercourse_ductbank_01xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_02xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_03xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_04xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_05xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_07xC")

mask_ac_access = (df['layer'] == 'EMF_bt_access_ductbank_01xC') | (df['layer'] == 'EMF_bt_access_ductbank_02xC') | (df['layer'] == 'EMF_bt_access_ductbank_03xC') | (df['layer'] == 'EMF_bt_access_ductbank_04xC') | (df['layer'] == 'EMF_bt_access_ductbank_05xC') | (df['layer'] == 'EMF_bt_access_ductbank_06xC') | (df['layer'] == 'EMF_bt_access_ductbank_07xC') | (df['layer'] == 'EMF_bt_access_ductbank_08xC')

mask_dc_cables = (df['layer'] == "EMF_cabo_solar_negativo") | (df['layer'] == "EMF_cabo_solar_positivo")

mask_dc_ducts = (df['layer'] == 'EMF_bt_eletroduto_1S') | (df['layer'] == 'EMF_bt_eletroduto_2S') | (df['layer'] == 'EMF_bt_eletroduto_3S')

mask_dc_trenches = ~(mask_dc_cables | mask_ac_cables | mask_ac_watercourse | mask_ac_access | mask_dc_ducts)

QTY. OF AC CABLES (m)
=

In [23]:
ac_cables_df = df[mask_ac_cables]
print(ac_cables_df)

                      layer  name       start_x        start_y         end_x  \
6858  EMF_bt_vala_btca_01xC  None  36054.691406  129707.171875  35947.617188   
6859  EMF_bt_vala_btca_01xC  None  33680.183594  129012.406250  33680.183594   
6860  EMF_bt_vala_btca_01xC  None  33699.042969  128996.218750  33699.066406   
6861  EMF_bt_vala_btca_01xC  None  33730.542969  128996.218750  33730.566406   
6862  EMF_bt_vala_btca_01xC  None  33768.367188  128996.218750  33768.367188   
...                     ...   ...           ...            ...           ...   
7959  EMF_bt_vala_btca_01xC  None  36235.281250  128949.523438  36213.242188   
7960  EMF_bt_vala_btca_01xC  None  36213.242188  128949.523438  36197.484375   
7961  EMF_bt_vala_btca_01xC  None  36197.484375  128948.523438  36197.484375   
7962  EMF_bt_vala_btca_01xC  None  36235.281250  128949.523438  36235.281250   
7963  EMF_bt_vala_btca_01xC  None  36159.699219  128775.156250  36159.699219   

              end_y      length  
6858 

In [24]:
ac_cables_group = ac_cables_df.groupby(by=['layer']).sum().reset_index()
print(ac_cables_group[['layer','length']])

                   layer        length
0  EMF_bt_vala_btca_01xC  11612.351562
1  EMF_bt_vala_btca_02xC   7535.069336
2  EMF_bt_vala_btca_03xC   5263.375977
3  EMF_bt_vala_btca_04xC   4043.333008
4  EMF_bt_vala_btca_05xC   3160.560303
5  EMF_bt_vala_btca_06xC   2192.055420
6  EMF_bt_vala_btca_07xC   1411.144897
7  EMF_bt_vala_btca_08xC   1724.205688


QTY. OF AC DUCTS IN WATERCOURSE (m and un.)
=

In [25]:
ac_watercourse_df = df[mask_ac_watercourse]
print(ac_watercourse_df)

                                 layer  name       start_x        start_y  \
7206  EMF_bt_watercourse_ductbank_05xC  None  33898.320312  128997.718750   
7207  EMF_bt_watercourse_ductbank_03xC  None  34134.812500  128777.039062   
7208  EMF_bt_watercourse_ductbank_02xC  None  34118.691406  128737.578125   
7209  EMF_bt_watercourse_ductbank_02xC  None  35990.718750  128880.648438   
7210  EMF_bt_watercourse_ductbank_04xC  None  35750.203125  129466.679688   
7252  EMF_bt_watercourse_ductbank_07xC  None  34951.957031  129537.664062   
7744  EMF_bt_watercourse_ductbank_04xC  None  34448.488281  127372.695312   
7915  EMF_bt_watercourse_ductbank_01xC  None  34524.824219  126221.343750   
7916  EMF_bt_watercourse_ductbank_01xC  None  34395.250000  127222.960938   
7917  EMF_bt_watercourse_ductbank_07xC  None  35281.406250  127268.187500   
7918  EMF_bt_watercourse_ductbank_03xC  None  35692.507812  127268.257812   
7919  EMF_bt_watercourse_ductbank_01xC  None  34826.191406  127501.609375   

In [26]:
ac_watercourse_group = ac_watercourse_df.groupby(by=['layer']).sum().reset_index()
print(ac_watercourse_group[['layer','length']])

                              layer     length
0  EMF_bt_watercourse_ductbank_01xC  65.981003
1  EMF_bt_watercourse_ductbank_02xC  44.042755
2  EMF_bt_watercourse_ductbank_03xC  44.005550
3  EMF_bt_watercourse_ductbank_04xC  44.000000
4  EMF_bt_watercourse_ductbank_05xC  22.903540
5  EMF_bt_watercourse_ductbank_07xC  44.004562


In [27]:
print(ac_watercourse_df['layer'].value_counts())

layer
EMF_bt_watercourse_ductbank_01xC    3
EMF_bt_watercourse_ductbank_03xC    2
EMF_bt_watercourse_ductbank_02xC    2
EMF_bt_watercourse_ductbank_04xC    2
EMF_bt_watercourse_ductbank_07xC    2
EMF_bt_watercourse_ductbank_05xC    1
Name: count, dtype: int64


QTY. OF AC DUCTS IN ACCESS (m and un.)
=

In [28]:
ac_access_df = df[mask_ac_access]
print(ac_access_df)

                            layer  name       start_x        start_y  \
7301  EMF_bt_access_ductbank_07xC  None  33777.882812  129007.320312   
7302  EMF_bt_access_ductbank_02xC  None  34572.292969  129115.382812   
7303  EMF_bt_access_ductbank_01xC  None  34614.230469  128840.960938   
7304  EMF_bt_access_ductbank_02xC  None  34604.304688  128663.773438   
7305  EMF_bt_access_ductbank_04xC  None  34619.199219  128646.429688   
7306  EMF_bt_access_ductbank_01xC  None  33985.890625  128997.718750   
7310  EMF_bt_access_ductbank_05xC  None  34614.226562  128511.875000   
7311  EMF_bt_access_ductbank_03xC  None  35063.558594  128531.039062   
7312  EMF_bt_access_ductbank_07xC  None  35126.148438  129030.664062   
7313  EMF_bt_access_ductbank_05xC  None  35403.226562  129037.078125   
7314  EMF_bt_access_ductbank_01xC  None  35133.375000  129260.007812   
7315  EMF_bt_access_ductbank_07xC  None  35139.375000  129504.515625   
7316  EMF_bt_access_ductbank_01xC  None  35151.007812  129533.07

In [29]:
ac_access_group = ac_access_df.groupby(by=['layer']).sum().reset_index()
print(ac_access_group[['layer','length']])

                         layer     length
0  EMF_bt_access_ductbank_01xC  75.325142
1  EMF_bt_access_ductbank_02xC  45.619965
2  EMF_bt_access_ductbank_03xC  31.175095
3  EMF_bt_access_ductbank_04xC  28.202538
4  EMF_bt_access_ductbank_05xC  42.000008
5  EMF_bt_access_ductbank_06xC  38.730659
6  EMF_bt_access_ductbank_07xC  27.352970
7  EMF_bt_access_ductbank_08xC  36.366802


In [30]:
print(ac_access_df['layer'].value_counts())

layer
EMF_bt_access_ductbank_01xC    8
EMF_bt_access_ductbank_05xC    7
EMF_bt_access_ductbank_02xC    5
EMF_bt_access_ductbank_03xC    5
EMF_bt_access_ductbank_08xC    5
EMF_bt_access_ductbank_07xC    4
EMF_bt_access_ductbank_06xC    4
EMF_bt_access_ductbank_04xC    3
Name: count, dtype: int64


QTY. OF DC CABLES (m)
=

In [31]:
dc_cables_df = df[mask_dc_cables]
print(dc_cables_df)

                         layer  name       start_x        start_y  \
14654  EMF_cabo_solar_positivo  None  36016.906250  129793.609375   
14655  EMF_cabo_solar_positivo  None  36016.906250  129751.109375   
14656  EMF_cabo_solar_positivo  None  36023.207031  129793.609375   
14657  EMF_cabo_solar_positivo  None  36023.207031  129751.109375   
14658  EMF_cabo_solar_positivo  None  36029.503906  129793.609375   
...                        ...   ...           ...            ...   
40732  EMF_cabo_solar_negativo  None  34400.085938  126040.796875   
40733  EMF_cabo_solar_negativo  None  34400.085938  125998.296875   
40734  EMF_cabo_solar_negativo  None  34406.386719  126040.796875   
40735  EMF_cabo_solar_negativo  None  34406.386719  125998.296875   
40736  EMF_cabo_solar_negativo  None  34412.687500  125998.296875   

              end_x          end_y      length  
14654  36054.171875  129708.742188  121.263138  
14655  36054.171875  129708.742188   78.761131  
14656  36054.171875  129

In [32]:
dc_cables_group = dc_cables_df.groupby(by=['layer']).sum().reset_index()
print(dc_cables_group[['layer','length']])
print(f'\nThe total length of DC cables is {round(dc_cables_group['length'].sum(),2)} meters')

                     layer        length
0  EMF_cabo_solar_negativo  1.069016e+06
1  EMF_cabo_solar_positivo  9.798206e+05

The total length of DC cables is 2048836.375 meters


QTY. OF DC DUCTS (m and un.)
=

In [33]:
dc_ducts_df =  df[mask_dc_ducts]
print(dc_ducts_df)

                      layer  name       start_x        start_y         end_x  \
10508  EMF_bt_eletroduto_2S  None  36016.906250  129710.070312  36054.171875   
10509  EMF_bt_eletroduto_2S  None  36023.207031  129710.070312  36054.171875   
10510  EMF_bt_eletroduto_2S  None  36029.503906  129710.070312  36054.171875   
10511  EMF_bt_eletroduto_2S  None  36035.804688  129710.070312  36054.171875   
10512  EMF_bt_eletroduto_2S  None  36042.105469  129710.070312  36054.171875   
...                     ...   ...           ...            ...           ...   
33900  EMF_bt_eletroduto_1S  None  34242.589844  125998.281250  34286.152344   
33901  EMF_bt_eletroduto_1S  None  34248.890625  125998.281250  34286.152344   
33902  EMF_bt_eletroduto_1S  None  34255.191406  125998.281250  34286.152344   
33903  EMF_bt_eletroduto_1S  None  34261.488281  125998.281250  34286.152344   
33904  EMF_bt_eletroduto_1S  None  34412.687500  125998.296875  34369.117188   

               end_y     length  
10508

In [34]:
dc_ducts_group = dc_ducts_df.groupby(by=['layer']).sum().reset_index()
print(dc_ducts_group[['layer','length']])

                  layer        length
0  EMF_bt_eletroduto_1S  41748.203125
1  EMF_bt_eletroduto_2S  31690.800781
2  EMF_bt_eletroduto_3S  36640.671875


In [35]:
print(dc_ducts_df['layer'].value_counts())
print(f'\nThe total quantity of DC ducts is {len(dc_ducts_df)}')

layer
EMF_bt_eletroduto_3S    2748
EMF_bt_eletroduto_2S    1378
EMF_bt_eletroduto_1S    1368
Name: count, dtype: int64

The total quantity of DC ducts is 5494


QTY. OF DC TRENCHES (m)
=

In [36]:
dc_trenches_df = df[mask_dc_trenches]
print(dc_trenches_df)

                 layer  name       start_x        start_y         end_x  \
0       EMF_vala_CC_2C  None  34683.585938  126169.992188  34685.281250   
1       EMF_vala_CC_4C  None  34685.281250  126171.421875  34689.886719   
2       EMF_vala_CC_1C  None  34757.492188  126242.078125  34752.886719   
3       EMF_vala_CC_3C  None  34536.992188  126293.210938  34533.984375   
4       EMF_vala_CC_2C  None  34798.683594  126336.312500  34804.984375   
...                ...   ...           ...            ...           ...   
10503  EMF_vala_CC_20C  None  34870.894531  126773.007812  34864.593750   
10504   EMF_vala_CC_7C  None  34458.484375  126125.000000  34463.085938   
10505  EMF_vala_CC_21C  None  34823.886719  126773.007812  34827.960938   
10506  EMF_vala_CC_15C  None  34458.484375  126045.968750  34462.554688   
10507  EMF_vala_CC_23C  None  34864.593750  126773.007812  34860.519531   

               end_y    length  
0      126171.421875  2.214942  
1      126171.140625  4.611770  


In [37]:
dc_trenches_group = dc_trenches_df.groupby(by=['layer']).sum().reset_index()
print(dc_trenches_group[['layer','length']])
print(f'\nThe total length of DC trenches is {round(dc_trenches_group['length'].sum(),2)} meters')

              layer       length
0   EMF_vala_CC_10C  1440.649902
1   EMF_vala_CC_11C   970.922485
2   EMF_vala_CC_12C  2135.308594
3   EMF_vala_CC_13C   583.526672
4   EMF_vala_CC_14C   406.086884
5   EMF_vala_CC_15C   265.987427
6   EMF_vala_CC_16C    52.206284
7   EMF_vala_CC_17C    64.544914
8   EMF_vala_CC_18C    12.213819
9   EMF_vala_CC_19C     4.074550
10   EMF_vala_CC_1C  3694.786377
11  EMF_vala_CC_20C     6.300000
12  EMF_vala_CC_21C     4.074550
13  EMF_vala_CC_23C     4.074392
14   EMF_vala_CC_2C  5324.682617
15   EMF_vala_CC_3C  7273.210938
16   EMF_vala_CC_4C  4043.126465
17   EMF_vala_CC_5C  1963.380859
18   EMF_vala_CC_6C  3588.523193
19   EMF_vala_CC_7C  1465.702637
20   EMF_vala_CC_8C  1943.108765
21   EMF_vala_CC_9C  2799.308594

The total length of DC trenches is 38045.80078125 meters


OVERVIEW
=

In [38]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\BOQ.xlsx") as xlsx:
    ac_cables_group[['layer','length']].to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    ac_watercourse_group[['layer','length']].to_excel(xlsx, sheet_name='AC Ducts Watercouse (m)', index=False)
    ac_watercourse_df['layer'].value_counts().to_excel(xlsx, sheet_name='AC Ducts Watercouse (un.)', index=False)
    ac_access_group[['layer','length']].to_excel(xlsx, sheet_name='AC Ducts Access (m)', index=False)
    ac_access_df['layer'].value_counts().to_excel(xlsx, sheet_name='AC Ducts Access (un.)', index=False)
    dc_cables_group[['layer','length']].to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group[['layer','length']].to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_df['layer'].value_counts().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group[['layer','length']].to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)


Pandas process took 4.582173585891724 seconds
